In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

df = pd.read_csv("glof_dataset.csv")
df.head()

In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
df.describe()

In [ ]:
print("Duplicate rows before cleaning:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Duplicate rows after cleaning:", df.duplicated().sum())

print("\nMissing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Handle missing values: numeric columns -> median imputation (robust to outliers)
for col in ["snowfall_mm", "earthquake_magnitude"]:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

print("Missing values remaining:", df.isnull().sum().sum())

In [ ]:
# Encode categorical column (region)
df["region_encoded"] = df["region"].astype("category").cat.codes
region_map = dict(enumerate(df["region"].astype("category").cat.categories))
print("Region encoding:", region_map)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
num_cols = ["lake_area_km2", "glacier_retreat_m_per_yr", "distance_from_glacier_m",
            "slope_deg", "rainfall_mm", "elevation_m"]
for ax, col in zip(axes.flat, num_cols):
    sns.histplot(df[col], kde=True, ax=ax, color="#2E86AB")
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
corr_cols = ["elevation_m", "lake_area_km2", "glacier_retreat_m_per_yr",
             "distance_from_glacier_m", "slope_deg", "rainfall_mm",
             "temperature_c", "snowfall_mm", "earthquake_magnitude", "glof_risk"]
corr = df[corr_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, col in zip(axes, ["lake_area_km2", "glacier_retreat_m_per_yr", "rainfall_mm"]):
    sns.boxplot(x="glof_risk", y=col, data=df, ax=ax, palette=["#4CAF6D", "#E84B4B"])
    ax.set_title(f"{col} by risk class")
plt.tight_layout()
plt.show()

In [ ]:
# Outlier detection using IQR method
def iqr_outliers(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    return ((series < lower) | (series > upper)).sum()

for col in num_cols:
    print(f"{col}: {iqr_outliers(df[col])} outliers (IQR method)")

In [ ]:
print("Class balance (glof_risk):")
print(df["glof_risk"].value_counts())
print(df["glof_risk"].value_counts(normalize=True).round(3))

In [ ]:
df["risk_score_proxy"] = (
    df["lake_area_km2"].rank(pct=True) * 0.3 +
    df["glacier_retreat_m_per_yr"].rank(pct=True) * 0.25 +
    (1 - df["distance_from_glacier_m"].rank(pct=True)) * 0.2 +
    df["rainfall_mm"].rank(pct=True) * 0.15 +
    df["slope_deg"].rank(pct=True) * 0.1
)

feature_cols = ["elevation_m", "lake_area_km2", "glacier_retreat_m_per_yr",
                 "distance_from_glacier_m", "slope_deg", "rainfall_mm",
                 "temperature_c", "snowfall_mm", "earthquake_magnitude", "region_encoded"]

X = df[feature_cols]
y = df["glof_risk"]
print("Feature matrix:", X.shape, " Target:", y.shape)

In [ ]:
# Hold the 10 real lakes out completely. They never enter training,
# validation, or model selection — only synthetic rows are used for that.
# This way validation/test metrics aren't inflated by the fact that the
# synthetic labels were generated from a formula built on these same features.
synthetic_mask = df["lake_type"] == "synthetic"
real_mask = df["lake_type"] == "real"

X_synth, y_synth = X[synthetic_mask], y[synthetic_mask]
X_real, y_real = X[real_mask], y[real_mask]

X_train, X_temp, y_train, y_temp = train_test_split(
    X_synth, y_synth, test_size=0.30, random_state=42, stratify=y_synth
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Train:", X_train.shape, " Val:", X_val.shape, " Test:", X_test.shape)
print("Held-out REAL lakes (never trained/tuned on):", X_real.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42),
}

results = []
trained_models = {}

for name, model in models.items():
    if name == "Logistic Regression":
        model.fit(X_train_s, y_train)
        preds = model.predict(X_val_s)
        proba = model.predict_proba(X_val_s)[:, 1]
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        proba = model.predict_proba(X_val)[:, 1]

    trained_models[name] = model
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, preds),
        "Precision": precision_score(y_val, preds),
        "Recall": recall_score(y_val, preds),
        "F1 Score": f1_score(y_val, preds),
        "ROC-AUC": roc_auc_score(y_val, proba),
    })

results_df = pd.DataFrame(results).sort_values("F1 Score", ascending=False).reset_index(drop=True)
results_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
results_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]].plot(
    kind="bar", ax=ax, colormap="viridis"
)
plt.title("Model Comparison on Validation Set")
plt.ylabel("Score")
plt.xticks(rotation=20)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_name]
print("Best model (by validation F1):", best_name)

if best_name == "Logistic Regression":
    X_test_input = scaler.transform(X_test)
else:
    X_test_input = X_test

test_preds = best_model.predict(X_test_input)
test_proba = best_model.predict_proba(X_test_input)[:, 1]

print("\n--- Test Set Performance ---")
print("Accuracy: ", round(accuracy_score(y_test, test_preds), 3))
print("Precision:", round(precision_score(y_test, test_preds), 3))
print("Recall:   ", round(recall_score(y_test, test_preds), 3))
print("F1 Score: ", round(f1_score(y_test, test_preds), 3))
print("ROC-AUC:  ", round(roc_auc_score(y_test, test_proba), 3))

### Real-lake generalization check

The metrics above are on a held-out slice of *synthetic* data, so they mostly confirm the model can recover its own label-generating formula. The real test of whether this generalizes is performance on the 10 real, named Himalayan lakes that were excluded from training and tuning entirely.

In [ ]:
# Evaluate the chosen model on the 10 real lakes it has never seen.
if best_name == "Logistic Regression":
    X_real_input = scaler.transform(X_real)
else:
    X_real_input = X_real

real_preds = best_model.predict(X_real_input)
real_proba = best_model.predict_proba(X_real_input)[:, 1]

real_results = df.loc[real_mask, ["lake_name", "region", "glof_risk"]].copy()
real_results["predicted_risk"] = real_preds
real_results["predicted_proba"] = real_proba.round(3)
real_results["correct"] = real_results["glof_risk"] == real_results["predicted_risk"]

acc_real = real_results["correct"].mean()
print(f"Accuracy on real lakes: {acc_real:.2%} ({real_results['correct'].sum()}/{len(real_results)})")
real_results.sort_values("predicted_proba", ascending=False)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, test_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Low Risk", "High Risk"], yticklabels=["Low Risk", "High Risk"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.show()

In [ ]:
if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
    plt.figure(figsize=(8, 5))
    sns.barplot(x=importances.values, y=importances.index, palette="mako")
    plt.title(f"Feature Importance — {best_name}")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()
else:
    coefs = pd.Series(best_model.coef_[0], index=feature_cols).sort_values()
    plt.figure(figsize=(8, 5))
    sns.barplot(x=coefs.values, y=coefs.index, palette="mako")
    plt.title(f"Coefficients — {best_name}")
    plt.tight_layout()
    plt.show()

In [ ]:
import joblib

joblib.dump(best_model, "glof_model.joblib")
joblib.dump(scaler, "glof_scaler.joblib")
joblib.dump(feature_cols, "glof_feature_cols.joblib")
joblib.dump(region_map, "glof_region_map.joblib")

print("Saved model:", best_name)
print("Files written: glof_model.joblib, glof_scaler.joblib, glof_feature_cols.joblib, glof_region_map.joblib")
